In [4]:
!pip install datasets

   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   ---------------------------------------- 604.4/604.4 kB 14.6 MB/s  0:00:00
   ---------------------------------------- 0.0/839.9 kB ? eta -:--:--
   ---------------------------------------- 839.9/839.9 kB 26.9 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 48.1 MB/s  0:00:00
   ---------------------------------------- 0.0/28.5 MB ? eta -:--:--
   ------------------- -------------------- 13.9/28.5 MB 69.5 MB/s eta 0:00:01
   ---------------------------------------  28.3/28.5 MB 74.0 MB/s eta 0:00:01
   ---------------------------------------- 28.5/28.5 MB 59.5 MB/s  0:00:00

   --- ------------------------------------  1/11 [pyarrow]
   --- ------------------------------------  1/11 [pyarrow]
   --- ------------------------------------  1/11 [pyarrow]
   --- ------------------------------------  1/11 [pyarrow]
   --- -------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.22.0 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
transformers 4.57.3 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 2.2.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import json
from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [10]:
def ConvertJsonToList(fileName, subject_name):
  subject_fileDataset = load_dataset("json", data_files=fileName)

  #print(subject1_fileDataset["train"]["metadata"])
  #print(len(subject1_fileDataset["train"]))
  samples = []

  for i in range (0,len(subject_fileDataset["train"])): #len(subject1)
    hashed_id = subject_fileDataset["train"][i]['hashed_id']
    keywords = subject_fileDataset["train"][i]['metadata']['keywords']
    mark = subject_fileDataset["train"][i]['nota']
    question = subject_fileDataset["train"][i]['metadata']['enunciado']
    responseStudent = subject_fileDataset["train"][i]['respuesta']
    responseTeacher = ""
    for j in range(0,len(subject_fileDataset["train"][i]['metadata']['minipreguntas'])):
      responseTeacher = responseTeacher + subject_fileDataset["train"][i]['metadata']['minipreguntas'][j]['minirespuesta']
    
    ie = {'question': question,
          'asignatura': subject_name,
          'responseTeacher': responseTeacher,
          'responseStudent': responseStudent,
          'mark': mark,
          'hashed_id': hashed_id,
          'keywords': keywords
          }

    samples.append(ie)

  return samples

In [11]:

#Creating a list with the necesarry fields
subjectFileList = ConvertJsonToList(
    "Datasets\\biConNotaAnon_Original.json",
    "Inteligencia Artificial e Ingeniería del Conocimiento"
)
subjectFileList2 = ConvertJsonToList(
    "Datasets\\metodos-de-captura-conNota-Anon _2_Original.json",
    "Bases de Datos para Big Data"
)
subjectFileList3 = ConvertJsonToList(
    "Datasets\\metodos-de-captura-conNota-Anon _3_Original.json",
    "Bases de Datos para Big Data"
)

subjectFileList = subjectFileList + subjectFileList2 + subjectFileList3

#Splitting the dataset into train,valid and test data
data_train ,data_test = train_test_split(subjectFileList,test_size=0.3)
data_train ,data_valid = train_test_split(data_train,test_size=0.1)

from json import encoder
data = {'train': data_train
     ,'test': data_test
     ,'valid': data_valid
     }

with open('Datasets\\JoinedSubjects.json', 'w', encoding = "utf-8") as outfile:
    json.dump(data, outfile, ensure_ascii=False)

In [12]:
import pandas as pd

with open("Datasets\\JoinedSubjects.json", "r", encoding="utf-8") as file:
    dataset = json.load(file)

def calculate_metrics(samples):
    marks = pd.to_numeric(
        pd.Series([sample.get("mark") for sample in samples]),
        errors="coerce"
    ).dropna()
    mark_stats = marks.describe()

    questions = {
        str(sample.get("question", sample.get("enunciado", ""))).strip()
        for sample in samples
        if sample.get("question", sample.get("enunciado"))
    }

    student_responses = {
        str(sample.get("responseStudent", "")).strip()
        for sample in samples
        if sample.get("responseStudent")
    }

    keywords = set()
    for sample in samples:
        values = sample.get("keywords", [])
        if isinstance(values, str):
            values = [values]
        if isinstance(values, list):
            keywords.update(str(value).strip() for value in values if value)

    return {
        "muestras": len(samples),
        "nota_min": mark_stats["min"] if not marks.empty else None,
        "nota_q1": mark_stats["25%"] if not marks.empty else None,
        "nota_media": mark_stats["mean"] if not marks.empty else None,
        "nota_mediana": mark_stats["50%"] if not marks.empty else None,
        "nota_q3": mark_stats["75%"] if not marks.empty else None,
        "nota_max": mark_stats["max"] if not marks.empty else None,
        "nota_iqr": mark_stats["75%"] - mark_stats["25%"] if not marks.empty else None,
        "nota_desviacion": mark_stats["std"] if not marks.empty else None,
    }

metrics = {
    split: calculate_metrics(samples)
    for split, samples in dataset.items()
    if isinstance(samples, list)
}

all_samples = [
    sample
    for split, samples in dataset.items()
    if isinstance(samples, list)
    for sample in samples
]
metrics["total"] = calculate_metrics(all_samples)

pd.DataFrame.from_dict(metrics, orient="index").round(2)

,muestras,nota_min,nota_q1,nota_media,nota_mediana,nota_q3,nota_max,nota_iqr,nota_desviacion
train,151,0.0,0.60,0.71,0.75,0.90,1.0,0.30,0.28
test,72,0.0,0.69,0.74,0.75,0.95,1.0,0.26,0.26
valid,17,0.0,0.65,0.71,0.75,0.95,1.0,0.30,0.27
total,240,0.0,0.60,0.72,0.75,0.95,1.0,0.35,0.27


In [13]:
samples_by_subject = {}

for sample in all_samples:
    subject = str(sample.get("asignatura", "")).strip()
    subject = subject or "Sin asignatura"
    samples_by_subject.setdefault(subject, []).append(sample)

metrics_by_subject = {
    subject: calculate_metrics(samples)
    for subject, samples in samples_by_subject.items()
}

metrics_by_subject_df = (
    pd.DataFrame.from_dict(metrics_by_subject, orient="index")
    .rename_axis("asignatura")
    .reset_index()
    .round(2)
)

metrics_by_subject_df

,asignatura,muestras,nota_min,nota_q1,nota_media,nota_mediana,nota_q3,nota_max,nota_iqr,nota_desviacion
0,Bases de Datos para Big Data,144,0.0,0.60,0.69,0.75,0.9,1.0,0.30,0.30
1,Inteligencia Artificial e Ingeniería del Conoc...,96,0.0,0.72,0.75,0.75,1.0,1.0,0.28,0.22
